In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [2]:
# input_texts = ["hi", "hello", "bye", "thanks", "yes", "no"]
# target_texts = ["hola", "hola", "adios", "gracias", "si", "no"]
# input_texts += ["good", "morning", "night", "please", "sorry"]
# target_texts += ["bueno", "mañana", "noche", "porfavor", "lo siento"]
# input_texts += ["friend", "food", "water", "house", "school", "book"]
# target_texts += ["amigo", "comida", "agua", "casa", "escuela", "libro"]
# print("Input texts :", input_texts)
# print("Target texts:", target_texts)

input_texts = [
    "hi", "hello", "bye", "thanks", "yes", "no",
    "good", "morning", "night", "please", "sorry",
    "friend", "food", "water", "house", "school", "book"
]

target_texts = [
    "hola", "saludo", "adios", "gracias", "si", "no",
    "bueno", "mañana", "noche", "porfavor", "lo siento",
    "amigo", "comida", "agua", "casa", "escuela", "libro"
]

# force better learning (important)
input_texts = input_texts * 20
target_texts = target_texts * 20

print("Input texts :", input_texts)
print("Target texts:", target_texts)

Input texts : ['hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food', 'water', 'house', 'school', 'book', 'hi', 'hello', 'bye', 'thanks', 'yes', 'no', 'good', 'morning', 'night', 'please', 'sorry', 'friend', 'food',

In [3]:
target_texts = ["\t" + text + "\n" for text in target_texts]

print("Target texts with start/end symbols:")
for t in target_texts:
    print(repr(t))

Target texts with start/end symbols:
'\thola\n'
'\tsaludo\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tmañana\n'
'\tnoche\n'
'\tporfavor\n'
'\tlo siento\n'
'\tamigo\n'
'\tcomida\n'
'\tagua\n'
'\tcasa\n'
'\tescuela\n'
'\tlibro\n'
'\thola\n'
'\tsaludo\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tmañana\n'
'\tnoche\n'
'\tporfavor\n'
'\tlo siento\n'
'\tamigo\n'
'\tcomida\n'
'\tagua\n'
'\tcasa\n'
'\tescuela\n'
'\tlibro\n'
'\thola\n'
'\tsaludo\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tmañana\n'
'\tnoche\n'
'\tporfavor\n'
'\tlo siento\n'
'\tamigo\n'
'\tcomida\n'
'\tagua\n'
'\tcasa\n'
'\tescuela\n'
'\tlibro\n'
'\thola\n'
'\tsaludo\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tmañana\n'
'\tnoche\n'
'\tporfavor\n'
'\tlo siento\n'
'\tamigo\n'
'\tcomida\n'
'\tagua\n'
'\tcasa\n'
'\tescuela\n'
'\tlibro\n'
'\thola\n'
'\tsaludo\n'
'\tadios\n'
'\tgracias\n'
'\tsi\n'
'\tno\n'
'\tbueno\n'
'\tmañana\n'
'\tnoche\n'
'\tporfavor\n'
'\tlo s

In [4]:

input_chars = sorted(list(set("".join(input_texts))))
target_chars = sorted(list(set("".join(target_texts))))

input_char_to_index = {char: i + 1 for i, char in enumerate(input_chars)}
target_char_to_index = {char: i + 1 for i, char in enumerate(target_chars)}

input_index_to_char = {i: char for char, i in input_char_to_index.items()}
target_index_to_char = {i: char for char, i in target_char_to_index.items()}

num_encoder_tokens = len(input_chars) + 1
num_decoder_tokens = len(target_chars) + 1

print("Input characters :", input_chars)
print("Target characters:", target_chars)
print("Number of encoder tokens:", num_encoder_tokens)
print("Number of decoder tokens:", num_decoder_tokens)


Input characters : ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'y']
Target characters: ['\t', '\n', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'ñ']
Number of encoder tokens: 22
Number of decoder tokens: 24


In [5]:

max_encoder_seq_length = max(len(text) for text in input_texts)
max_decoder_seq_length = max(len(text) for text in target_texts)

print("Max encoder sequence length:", max_encoder_seq_length)
print("Max decoder sequence length:", max_decoder_seq_length)


Max encoder sequence length: 7
Max decoder sequence length: 11


In [6]:

encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length), dtype="int32")
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length), dtype="int32")
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # Encoder input
    for t, char in enumerate(input_text):
        encoder_input_data[i, t] = input_char_to_index[char]

    # Decoder input and decoder target
    for t, char in enumerate(target_text):
        decoder_input_data[i, t] = target_char_to_index[char]
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_index[char]] = 1.0

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)


Encoder input shape: (340, 7)
Decoder input shape: (340, 11)
Decoder target shape: (340, 11, 24)


In [7]:
latent_dim = 256

encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(input_dim=num_encoder_tokens, output_dim=64, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

encoder_states = [state_h, state_c]

In [8]:

decoder_inputs = Input(shape=(None,), name="decoder_inputs")
# decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=16, mask_zero=True, name="decoder_embedding")
decoder_embedding_layer = Embedding(input_dim=num_decoder_tokens, output_dim=64, mask_zero=True)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)


In [9]:

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 64)  │      1,408 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │      1,536 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 256),     │    328,704 │ encoder_embeddin… │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    328,704 │ embedding[0][0],  │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256), (None,      │            │ encoder_lstm[0][… │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 24)  │      6,168 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 666,520 (2.54 MB)

 Trainable params: 666,520 (2.54 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=500,
    verbose=1
)

Epoch 1/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.2415 - loss: 2.0777
Epoch 2/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.5492 - loss: 1.0692
Epoch 3/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.8190 - loss: 0.2647
Epoch 4/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.8544 - loss: 0.0958
Epoch 5/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.8621 - loss: 0.0413
Epoch 6/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.8613 - loss: 0.0340
Epoch 7/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.8629 - loss: 0.0239
Epoch 8/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.8629 - loss: 0.0204
Epoch 9/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.8629 - loss: 0.0171
Epoch 10/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.8629 - loss: 0.0152
Epoch 11/500
170/170 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.8629 - loss: 0.0140
Epoch 12/500
170/170 ━━━━━━━

In [11]:

# Encoder inference model
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_embedding_inf = decoder_embedding_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)


In [12]:

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1), dtype="int32")
    target_seq[0, 0] = target_char_to_index["\t"]

    decoded_sentence = ""

    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_index_to_char.get(sampled_token_index, "")

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            break

        decoded_sentence += sampled_char

        target_seq = np.zeros((1, 1), dtype="int32")
        target_seq[0, 0] = sampled_token_index

        states_value = [h, c]

    return decoded_sentence


In [ ]:

for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input: {input_texts[seq_index]}  -->  Predicted Output: {decoded_sentence}")


Input: hi  -->  Predicted Output: hola
Input: hello  -->  Predicted Output: saludo
Input: bye  -->  Predicted Output: adios
Input: thanks  -->  Predicted Output: gracias
Input: yes  -->  Predicted Output: si
Input: no  -->  Predicted Output: no
Input: good  -->  Predicted Output: bueno
Input: morning  -->  Predicted Output: mañana
Input: night  -->  Predicted Output: noche
Input: please  -->  Predicted Output: porfavor
Input: sorry  -->  Predicted Output: lo siento
Input: friend  -->  Predicted Output: amigo
Input: food  -->  Predicted Output: comida
Input: water  -->  Predicted Output: agua
Input: house  -->  Predicted Output: casa
Input: school  -->  Predicted Output: escuela
Input: book  -->  Predicted Output: libro
Input: hi  -->  Predicted Output: hola
Input: hello  -->  Predicted Output: saludo
Input: bye  -->  Predicted Output: adios
Input: thanks  -->  Predicted Output: gracias
Input: yes  -->  Predicted Output: si
Input: no  -->  Predicted Output: no
Input: good  -->  Predicte

In [ ]:

def encode_input_text(text):
    seq = np.zeros((1, max_encoder_seq_length), dtype="int32")
    for t, char in enumerate(text[:max_encoder_seq_length]):
        if char in input_char_to_index:
            seq[0, t] = input_char_to_index[char]
    return seq

# Examples
test_words = ["hi", "bye", "yes", "no", "hello", "thanks"]
for word in test_words:
    encoded = encode_input_text(word)
    print(f"Input: {word}  -->  Output: {decode_sequence(encoded)}")
